<h1 style="background: linear-gradient(to right, #395761, #7799B6);
           color: white;
           padding: 22px;
           border-left: 10px solid #CEEB13;
           text-align: center;
           font-family: 'Roboto Mono', Menlo, 'Courier New', monospace;
           font-weight: 600;">
  2. Tune SGLang
</h1>

You change one server setting at a time, redeploy, and run the same standard benchmark as your baseline, so every difference you see comes from that one setting. Each experiment goes on the results wall under its own label.

**Core, ≈ 8 min** (one redeploy: FP8):

| Step | What you do | Time |
|---|---|---|
| 0. Where you are | Read your current config and baseline from the cluster | 1 min |
| 1. FP8 KV cache | `kv_cache_dtype="fp8_e4m3"`: redeploy, benchmark, compare, submit | ~5½ min |
| 3. HiCache | Read why it needs a bigger machine than your slice; try it at home | read while you wait |
| 4. Summary | All your runs side by side | 1 min |

**Optional, ≈ 15 min**, if you have time:

| Step | What you do | Time |
|---|---|---|
| 2. Scheduling | `max_running_requests=128`: redeploy, benchmark, compare, submit | ≈ 5½ min |
| 5. Try your own settings | Pick values from a short menu, submit | ≈ 6 min |
| 6. Put the baseline back | Only if you want the baseline back | ≈ 3½ min |

Each redeploy stops your server, applies the change and starts it again (usually 3 to 3½ minutes; you have one GPU, so the old and new server cannot run side by side), and the standard benchmark takes about 2 minutes. The model download sometimes takes up to 10 minutes; the text under each experiment cell is there to read while it waits.

## 0. Where you are

This cell reads everything from the cluster, not from this notebook's memory: the configuration your server is running right now (the arguments of its pod), and the standard-benchmark runs you submitted, including the baseline from part 1. So it works whether you are on the baseline or half way through, and after a kernel restart: just run it again and continue with the next experiment you have not done.

In [ ]:
import sys

sys.path.insert(0, "..")

from workshop_utils import (
    PortForward,
    current_config,
    participant_config,
    participant_namespace,
    participant_profile,
    server_info,
    standard_bench,
    standard_results,
    submit,
    switch_to,
)
from workshop_utils.bench import compare_table
from workshop_utils.plots import compare_bars

NAMESPACE = participant_namespace()
prof = participant_profile()
base_cfg = participant_config(NAMESPACE)  # the baseline: SGLang defaults

print(current_config(NAMESPACE).describe())
runs = standard_results(NAMESPACE)  # your submitted standard runs, latest per label
baseline = runs.get("baseline")
print("submitted so far:", ", ".join(f"{k} ({r.output_throughput:.0f} tok/s)" for k, r in runs.items()) or "nothing")
if baseline is None:
    print("No baseline submitted yet: run the next cell once.")

In [ ]:
# Only needed if you did not submit a baseline in part 1 (or want to re-measure it): about 2 to 6 minutes.
if baseline is None:
    switch_to(prof, base_cfg)
    baseline = standard_bench(base_cfg.name, NAMESPACE, base_cfg.served_model_name, "baseline")
    submit(baseline, base_cfg, label="baseline")

**How a change is applied.** `switch_to(prof, cfg)` compares `cfg` with what is running. If they are the same it does nothing. Otherwise it calls `redeploy`, which follows the only route that works with one GPU: annotate the service `serving.kserve.io/stop=true` (KServe deletes the pod, your GPU is free), apply the new manifest, remove the annotation, and wait for the *new* pod, the one whose arguments are exactly the new configuration, to be Ready. A plain `kubectl apply` would start the new pod next to the old one, and it would wait forever for a second GPU.

## 1. FP8 KV cache

Every token in flight keeps its keys and values for every layer in GPU memory: for Qwen3-8B that is 36 layers × 8 KV heads × 128 dimensions × 2 (K and V) × 2 bytes in bf16, 144 KiB per token. `kv_cache_dtype="fp8_e4m3"` stores them in 1 byte instead of 2, so the same memory holds twice as many tokens: `max_total_num_tokens` goes from 803,502 to about 1.6 million.

That matters when the pool is full, and in the standard benchmark it is: 256 requests × (4096 + 512) tokens is about 1.18 million tokens, more than the bf16 pool holds. With bf16, roughly 170 requests fit and the rest wait in the queue; with FP8 all 256 fit.

Run the cell, then read on while it redeploys and benchmarks (about 5½ minutes). The model download sometimes takes up to 10 minutes; read the next section while it loads.

In [ ]:
exp1 = participant_config(NAMESPACE, kv_cache_dtype="fp8_e4m3")  # ONE change against the baseline
switch_to(prof, exp1)
r1 = standard_bench(exp1.name, NAMESPACE, exp1.served_model_name, "exp1-fp8")

**While it runs.** What to expect, from the rehearsal on one H200 (your GPU will differ by a few percent):

- **Capacity under load: +11% output throughput** (2,346 → 2,601 tok/s). More requests decode in each step, so the GPU does more useful work per weight read.
- **p99 time to first token halved** (53 s → 27.5 s): no request waits in the queue for memory any more.
- **Per-user speed goes down: time per output token +20%** (47 → 56 ms median). Each decode step now serves more sequences, so each one gets its next token a little later. More users served, each a little slower: the usual capacity trade.

Two baseline runs on the same GPU agreed within 0.5%, so an 11% change is the setting, not noise.

FP8 is not free: keys and values are rounded to 8 bits. For most chat workloads the quality change is hard to notice, but check it on your own evaluation before using it in production. On AMD the same flag works; the MI355X has 288 GB, so the pool is larger to begin with.

Confirm the flag took effect, then compare with your baseline:

In [ ]:
with PortForward(exp1.name, NAMESPACE) as url:
    info = server_info(url)
print("kv_cache_dtype:", info.get("kv_cache_dtype"), "  max_total_num_tokens:", info.get("max_total_num_tokens"))
keys = ("output_throughput", "median_ttft_ms", "p99_ttft_ms", "median_tpot_ms")
print(compare_table(baseline, [r1], keys=keys))
compare_bars({"baseline": baseline, "exp1-fp8": r1}, metrics=("output_throughput", "p99_ttft_ms", "median_tpot_ms"),
             title="FP8 KV cache vs baseline, standard benchmark");

In [ ]:
submit(r1, exp1, label="exp1-fp8");

## 2. Scheduling: cap the running batch (optional)

*Optional, about 5½ minutes. Short on time? Skip to [4. Summary](#4.-Summary).*

SGLang's scheduler admits a waiting request into the running batch as soon as there is memory for it. In the baseline that is about 170 of the 256 benchmark requests at once (as many as the bf16 pool holds), and every decode step serves all of them. `max_running_requests=128` caps the batch at 128: the other requests wait in the queue until a slot frees up, even when memory is free. This experiment starts from the baseline again (bf16), not from FP8.

This is the classic serving trade-off between how fast each user's answer streams and how many users you serve at once:

- **Time per output token (TPOT)** goes down: a decode step over 128 sequences is shorter than over 170, so each admitted user's text streams faster.
- **Time to first token (TTFT)** goes up: more requests wait in the queue before they start.
- **Throughput** changes little: the GPU is still busy with a large batch.

On the rehearsal GPU: median TPOT −14% (p99 −19%), median TTFT about doubled (16 s → 31 s), throughput −3%.

Run it and read on while it redeploys and benchmarks (about 5½ minutes). The model download sometimes takes up to 10 minutes; read the next section while it loads.

In [ ]:
exp2 = participant_config(NAMESPACE, max_running_requests=128)  # ONE change against the baseline (not on top of FP8)
switch_to(prof, exp2)
r2 = standard_bench(exp2.name, NAMESPACE, exp2.served_model_name, "exp2-max-running-128")

**While it runs.** Why would anyone cap the batch? Because the users you admit get a guaranteed streaming speed, which matters for interactive applications and for service-level objectives written as "p99 time per token under X ms". A related knob is `chunked_prefill_size` (8192 tokens by default on the H200): long prompts are prefilled in chunks of that size between decode steps, so one huge prompt cannot stall everyone else's decoding. With 4096-token prompts every prompt fits in one chunk, so it barely matters here, but it does with 32k-token documents. References: [hyperparameter tuning](https://docs.sglang.io/advanced_features/hyperparameter_tuning.html), [server arguments](https://docs.sglang.io/advanced_features/server_arguments.html).

In [ ]:
keys = ("output_throughput", "median_ttft_ms", "median_tpot_ms", "p99_tpot_ms")
print(compare_table(baseline, [r2], keys=keys))
compare_bars({"baseline": baseline, "exp2-max-running-128": r2}, metrics=keys,
             title="Capped batch vs baseline, standard benchmark");

In [ ]:
submit(r2, exp2, label="exp2-max-running-128");

This experiment will not raise your throughput, and that is the point: it shows what you trade when you optimize for the users who are already being served. Compare it with FP8, which moved the other way: more users at once, each a little slower.

## 3. HiCache: try it at home

[HiCache](https://docs.sglang.io/advanced_features/hicache_best_practices.html) extends the radix (prefix) cache from GPU memory to host memory, and optionally to storage. When many requests share long prefixes (a system prompt, a document, a multi-turn chat) and those prefixes no longer fit on the GPU, the evicted prefixes stay in host memory and are copied back instead of being recomputed.

It does not fit in a workshop slice, and here is why in numbers. Your GPU pool is about 110 GB of KV cache (803,502 tokens × 144 KiB). The host pool has to be **larger** than that to add anything, because with the default write-through policy it first holds copies of what is already on the GPU. Your pod may use at most 128 GiB of host memory in total, and SGLang sizes the host pool to 80% of what is left after a 10 GiB reserve: about 94 GiB, smaller than the GPU pool. The H200 nodes have 2 TB of RAM, so on a whole node, or with `mem_fraction_static` lowered so the GPU pool is smaller, HiCache has room to work.

To try it on your own GPU (one change, plus the memory it needs), on a shared-prefix workload, with the baseline measured on the same workload for comparison. These results are not the standard benchmark, so they are not ranked even if you submit them.

In [ ]:
TRY_HICACHE = False  # set to True on a machine whose pod can have more host memory than its GPU KV pool

if TRY_HICACHE:
    from workshop_utils import run_bench

    # 64 groups x 16k-token shared prefix = about 1M prefix tokens: more than the GPU pool holds.
    shared_prefix = dict(dataset="generated-shared-prefix", num_prompts=512, max_concurrency=32, timeout_s=1800, extra=[
        "--gsp-num-groups", "64", "--gsp-prompts-per-group", "8", "--gsp-system-prompt-len", "16384",
        "--gsp-question-len", "128", "--gsp-output-len", "64"])
    switch_to(prof, base_cfg)
    hb = run_bench(base_cfg.name, NAMESPACE, base_cfg.served_model_name, "baseline shared-prefix", **shared_prefix)
    exp3 = participant_config(NAMESPACE, hicache=True, page_size=64, hicache_size_gb=200, memory_gb_per_gpu=240)
    switch_to(prof, exp3)
    h3 = run_bench(exp3.name, NAMESPACE, exp3.served_model_name, "exp3-hicache shared-prefix", **shared_prefix)
    print(compare_table(hb, [h3]))

## 4. Summary

Everything you submitted, read back from the cluster and compared with your baseline:

In [ ]:
runs = standard_results(NAMESPACE)
baseline = runs.pop("baseline", baseline)
print(compare_table(baseline, runs.values(), keys=("output_throughput", "median_ttft_ms", "p99_ttft_ms", "median_tpot_ms", "p99_tpot_ms")))
compare_bars({"baseline": baseline, **runs}, title="Your experiments, standard benchmark");

- One `ServingConfig` field, one redeploy, the same benchmark: that is how you tune a serving engine without fooling yourself.
- FP8 KV cache doubles the token capacity of the same GPU memory. Under load, when the bf16 pool is full, that means more users served at once: more throughput and far less waiting, at a small cost in per-user streaming speed.
- Capping the running batch goes the other way: faster streaming for the users being served, longer waits for the rest.
- HiCache pays off when shared prefixes outgrow GPU memory and the machine has the host memory to hold them.

Further reading: [SGLang hyperparameter tuning](https://docs.sglang.io/advanced_features/hyperparameter_tuning.html), [server arguments](https://docs.sglang.io/advanced_features/server_arguments.html), [quantized KV cache](https://docs.sglang.io/advanced_features/quantized_kv_cache.html), [HiCache best practices](https://docs.sglang.io/advanced_features/hicache_best_practices.html), [benchmark and profiling](https://docs.sglang.io/developer_guide/benchmark_and_profiling.html).

That is the core of part 2. The two steps below are optional (and so is step 2, if you skipped it).

## 5. Try your own settings (optional)

Now combine settings yourself, for what you care about: throughput, time to first token, or streaming speed. Pick values from the menu below, run the cell, and it is submitted under the label `best`. The results wall shows which combinations people chose and how they did.

| Setting | Allowed | What it does on this benchmark |
|---|---|---|
| `kv_cache_dtype` | `None` (bf16) or `"fp8_e4m3"` | Experiment 1: twice the KV capacity, +11% throughput, p99 TTFT halved, TPOT +20%. |
| `max_running_requests` | `None` (no cap) or 128 to 256 | Experiment 2: a lower cap trades waiting time for faster streaming. With FP8, a cap of 192 kept p99 TPOT lower (74 vs 93 ms) for about 5 points less throughput. |
| `mem_fraction_static` | `None` (0.903) or 0.85 to 0.95 | The share of GPU memory for weights plus KV cache; more means a bigger pool and less room for activations. No gain measured on this load: try it and see. |

*Try at home:* `extra_args=("--enable-torch-compile",)` compiles the model with `torch.compile` for small batches. It can speed up decoding, but it adds minutes to every start, too long for this session. `max_prefill_tokens` (how many prompt tokens one prefill step may take) made no difference here.

The cell redeploys and benchmarks (about 5½ minutes). The model download sometimes takes up to 10 minutes; read the next section while it loads.

In [ ]:
mine = participant_config(
    NAMESPACE,
    kv_cache_dtype="fp8_e4m3",  # None or "fp8_e4m3"
    max_running_requests=None,  # None, or 128 to 256
    mem_fraction_static=None,  # None, or 0.85 to 0.95
)
assert mine.kv_cache_dtype in (None, "fp8_e4m3"), "kv_cache_dtype: None or 'fp8_e4m3'"
assert mine.max_running_requests is None or 128 <= mine.max_running_requests <= 256, "max_running_requests: 128 to 256"
assert mine.mem_fraction_static is None or 0.85 <= mine.mem_fraction_static <= 0.95, "mem_fraction_static: 0.85 to 0.95"

switch_to(prof, mine)
rb = standard_bench(mine.name, NAMESPACE, mine.served_model_name, "best")
print(compare_table(baseline, [rb], keys=("output_throughput", "median_ttft_ms", "p99_ttft_ms", "median_tpot_ms", "p99_tpot_ms")))
submit(rb, mine, label="best");  # re-run the summary cell (step 4) to see it next to the rest

## 6. Put the baseline back (optional)

Only if you want the baseline back. Your server keeps running whatever you deployed last, and that is fine to leave as it is. Run this to return to the SGLang defaults, for example before using the chat UI from part 1 again (≈ 3½ min; the model download sometimes takes up to 10 minutes):

In [ ]:
switch_to(prof, base_cfg)
print(current_config(NAMESPACE).describe())